# Make `fake_daily_rainfall` Training Data

This notebook documents how to generate the synthetic training dataset used by the extraction pipeline.

Environment requirement:
- `conda activate weather-doc-extractor`

## 1. Configure generation settings

Set these once, then rerun later cells.

In [42]:
from pathlib import Path
import sys

# Ensure notebook imports use the local repository source tree.
# This avoids stale site-packages versions shadowing current edits.
_repo_root = Path.cwd().resolve()
if not (_repo_root / "src").exists():
    _repo_root = _repo_root.parent
_src_path = str(_repo_root / "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)

# If an older installed package was already imported, clear it so imports reload from src.
for _name in list(sys.modules):
    if _name == "weather_doc_extractor" or _name.startswith("weather_doc_extractor."):
        del sys.modules[_name]

# Output root for generated synthetic data
OUTPUT_DIR = Path("../fake_daily_rainfall_2")

# Main dataset size
N_RECORDS = 1000

# Reproducibility
SEED = 42

# Rendering parameters
FONT_SIZE = 20.0
FONT_SIZE_JITTER = 0.22
JITTER_GRID_POINTS = 0.0008
JPEG_QUALITY = 85
RIGHT_DAY_LABEL_PROBABILITY = 0.9
POST_DEC_BLANK_COLUMN_PROBABILITY = 0.2
LINE_INTENSITY_SIGMA = 0.40
INDIVIDUAL_LINE_INTENSITY_SIGMA = 0.2

OUTPUT_DIR

PosixPath('../fake_daily_rainfall_2')

## 2. (Optional) Generate one sample pair first

This is a quick sanity check before generating the full dataset.

In [37]:
import importlib
import inspect

import weather_doc_extractor.make_fake_training_data.draw_grid as draw_grid_module
import weather_doc_extractor.make_fake_training_data.make_pair as make_pair_module
import weather_doc_extractor.make_fake_training_data.make_data as make_data_module

draw_grid_module = importlib.reload(draw_grid_module)
make_pair_module = importlib.reload(make_pair_module)
make_data_module = importlib.reload(make_data_module)

make_pair = make_pair_module.make_pair
make_stem = make_data_module.make_stem

# Uncomment for debugging if needed:
# print(inspect.getsourcefile(make_pair))
# print(inspect.signature(make_pair))

year = 1875
county = "Cornwall"
station_id = 59
stem = make_stem(year, county, station_id)

img_path, json_path = make_pair(
    stem=stem,
    output_dir=OUTPUT_DIR,
    year=year,
    county=county,
    station_id=station_id,
    seed=SEED,
    font_size=FONT_SIZE,
    jitter_grid_points=JITTER_GRID_POINTS,
    jpeg_quality=JPEG_QUALITY,
    right_day_label_probability=RIGHT_DAY_LABEL_PROBABILITY,
    post_dec_blank_column_probability=POST_DEC_BLANK_COLUMN_PROBABILITY,
    line_intensity_sigma=LINE_INTENSITY_SIGMA,
    individual_line_intensity_sigma=INDIVIDUAL_LINE_INTENSITY_SIGMA,
)

img_path, json_path

(PosixPath('../fake_daily_rainfall_2/images/DRain_1871-1880_Cornwall-59.jpg'),
 PosixPath('../fake_daily_rainfall_2/transcriptions/DRain_1871-1880_Cornwall-59.json'))

## 3. Generate full synthetic dataset

Creates image/JSON pairs under:
- `fake_daily_rainfall/images`
- `fake_daily_rainfall/transcriptions`

In [43]:
import importlib

import weather_doc_extractor.make_fake_training_data.draw_grid as draw_grid_module
import weather_doc_extractor.make_fake_training_data.make_datasets as make_datasets_module

draw_grid_module = importlib.reload(draw_grid_module)
make_datasets_module = importlib.reload(make_datasets_module)
make_fake_dataset = make_datasets_module.main

make_fake_dataset(
    n_records=N_RECORDS,
    output_dir=str(OUTPUT_DIR),
    seed=SEED,
    font_size=FONT_SIZE,
    font_size_jitter=FONT_SIZE_JITTER,
    jitter_grid_points=JITTER_GRID_POINTS,
    jpeg_quality=JPEG_QUALITY,
    right_day_label_probability=RIGHT_DAY_LABEL_PROBABILITY,
    post_dec_blank_column_probability=POST_DEC_BLANK_COLUMN_PROBABILITY,
    line_intensity_sigma=LINE_INTENSITY_SIGMA,
    individual_line_intensity_sigma=INDIVIDUAL_LINE_INTENSITY_SIGMA,
)

## 4. Verify counts and pairing

Checks that image and transcription stems match.

In [44]:
images_dir = OUTPUT_DIR / "images"
transcriptions_dir = OUTPUT_DIR / "transcriptions"

image_stems = {p.stem for p in images_dir.glob("*.jpg")}
json_stems = {p.stem for p in transcriptions_dir.glob("*.json")}

n_images = len(image_stems)
n_json = len(json_stems)
n_paired = len(image_stems & json_stems)
n_unpaired_images = len(image_stems - json_stems)
n_unpaired_json = len(json_stems - image_stems)

print(f"Images: {n_images}")
print(f"Transcriptions: {n_json}")
print(f"Paired stems: {n_paired}")
print(f"Unpaired image stems: {n_unpaired_images}")
print(f"Unpaired transcription stems: {n_unpaired_json}")

Images: 999
Transcriptions: 999
Paired stems: 999
Unpaired image stems: 0
Unpaired transcription stems: 0


## 5. Build test dataset (optional)

Creates `test_data/fake/{images,transcriptions}`. 
Same construction as main dataset, but smaller, and with different seed, so suitable for test & validation.

In [ ]:
import importlib

import weather_doc_extractor.make_fake_training_data.draw_grid as draw_grid_module
import weather_doc_extractor.make_fake_training_data.make_datasets as make_datasets_module

draw_grid_module = importlib.reload(draw_grid_module)
make_datasets_module = importlib.reload(make_datasets_module)
make_fake_dataset = make_datasets_module.main

make_fake_dataset(
    n_records=64,
    output_dir='../test_data/fake',
    seed=SEED+1,
    font_size=FONT_SIZE,
    font_size_jitter=FONT_SIZE_JITTER,
    jitter_grid_points=JITTER_GRID_POINTS,
    jpeg_quality=JPEG_QUALITY,
    right_day_label_probability=RIGHT_DAY_LABEL_PROBABILITY,
    post_dec_blank_column_probability=POST_DEC_BLANK_COLUMN_PROBABILITY,
    line_intensity_sigma=LINE_INTENSITY_SIGMA,
    individual_line_intensity_sigma=INDIVIDUAL_LINE_INTENSITY_SIGMA,
)

## 6. Upload to Azure

In [ ]:
%%bash
bash ../scripts/aml_delete.sh fake_daily_rainfall_2/images
bash ../scripts/aml_upload.sh --src ../fake_daily_rainfall_2/images --dst fake_daily_rainfall_2/images
bash ../scripts/aml_delete.sh fake_daily_rainfall_2/transcriptions
bash ../scripts/aml_upload.sh --src ../fake_daily_rainfall_2/transcriptions --dst fake_daily_rainfall_2/transcriptions
bash ../scripts/aml_delete.sh test_data/fake/images
bash ../scripts/aml_upload.sh --src ../test_data/fake/images --dst test_data/fake/images
